# 01 — EDA: Exploratory Data Analysis

Goal: Load the raw transaction data, clean it, explore spending patterns,
and save a processed monthly dataset ready for modeling.

## 1. Imports and Setup

We load all required libraries for data manipulation, visualization, 
and time series analysis. Plot styles and figure defaults are set 
globally so every chart in the notebook is consistent.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

from statsmodels.tsa.seasonal import seasonal_decompose

sns.set_style("whitegrid")
warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (12, 5)

## 2. Load Raw Data

We load both Kaggle CSVs separately and combine them into a single 
dataframe. The original Kaggle train/test split is **random**, not 
time-based — meaningless for forecasting. We combine everything now 
and will make our own chronological split in the modeling notebooks.

In [ ]:
df_train = pd.read_csv("../data/raw/fraudTrain.csv")
df_test  = pd.read_csv("../data/raw/fraudTest.csv")

print("Train shape:", df_train.shape)
print("Test shape: ", df_test.shape)

df = pd.concat([df_train, df_test], ignore_index=True)
print("Combined shape:", df.shape)

In [ ]:
print(df.columns.tolist())
print()
print(df.dtypes)
print()
df.head(5)

## 3. Clean Data

We remove fraudulent transactions (0.52% of data) since we are 
modeling legitimate spending behavior only. Ten irrelevant columns 
are dropped, dates are parsed and sorted chronologically, and age 
is engineered from date of birth. Final shape: **1,842,743 rows × 12 columns**.

In [ ]:
# See fraud split before removing
print("Fraud value counts:")
print(df['is_fraud'].value_counts())
print(f"Fraud rate: {df['is_fraud'].mean():.2%}")

# Filter to legitimate transactions only, then drop the label
df = df[df['is_fraud'] == 0]
df = df.drop(columns=['is_fraud'])

# Drop columns we don't need at any phase
cols_to_drop = [
    'Unnamed: 0',  # duplicate index
    'first',       # name, no predictive value
    'last',        # name, no predictive value
    'street',      # zip covers location
    'lat',         # zip covers location
    'long',        # zip covers location
    'trans_num',   # transaction ID, useless
    'unix_time',   # redundant with trans_date_trans_time
    'merch_lat',   # redundant with merchant/zip
    'merch_long',  # redundant with merchant/zip
]
df = df.drop(columns=cols_to_drop)

# Parse date and sort chronologically
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df = df.sort_values('trans_date_trans_time').reset_index(drop=True)

print("\nShape after cleaning:", df.shape)
print("\nColumns:", df.columns.tolist())
print()
print(df.dtypes)
print()
df.head(3)

## 4. Data Checks

Before exploring, we verify the dataset is what we expect: correct 
date range, no missing values, and sensible summary statistics on 
transaction amounts. Any surprises here would invalidate everything downstream.

In [ ]:
print("Date range:")
print("  Start:", df['trans_date_trans_time'].min())
print("  End:  ", df['trans_date_trans_time'].max())
print()
print("Unique users (cc_num):", df['cc_num'].nunique())
print("Unique categories:    ", df['category'].nunique())
print("Unique merchants:     ", df['merchant'].nunique())
print()
print("Missing values:")
print(df.isnull().sum())
print()
print("Amount stats:")
print(df['amt'].describe().round(2))

## 5. Feature Engineering — Age

We convert `dob` (date of birth) into a usable `age` column by 
computing the difference in days from each transaction date, then 
dividing by 365. Age is more interpretable and useful for modeling 
than a raw birth date. The `dob` column is dropped after.

In [ ]:
df['dob'] = pd.to_datetime(df['dob'])
df['age'] = (df['trans_date_trans_time'] - df['dob']).dt.days // 365
df = df.drop(columns=['dob'])

print("Age stats:")
print(df['age'].describe().round(1))
print()
df[['trans_date_trans_time', 'cc_num', 'amt', 'age']].head(3)

## 6. Category Breakdown

We examine how total spending is distributed across the 14 spending 
categories. This tells us which categories dominate platform volume 
and which are strong candidates for category-level forecasting later.

In [ ]:
category_totals = (
    df.groupby('category')['amt']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'amt': 'total_spending'})
)

category_totals['total_spending_fmt'] = category_totals['total_spending'].apply(lambda x: f"${x:,.0f}")
category_totals['pct_of_total'] = (category_totals['total_spending'] / category_totals['total_spending'].sum() * 100).round(2).astype(str) + '%'

print(category_totals[['category', 'total_spending_fmt', 'pct_of_total']].to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    category_totals['category'][::-1],
    category_totals['total_spending'][::-1],
    color='steelblue'
)

for bar, val in zip(bars, category_totals['total_spending'][::-1]):
    ax.text(bar.get_width() + 500000, bar.get_y() + bar.get_height()/2,
            f'${val:,.0f}', va='center', fontsize=9)

ax.set_title("Total Spending by Category (All Users, 2019–2020)", fontsize=13)
ax.set_xlabel("Total Amount ($)")
plt.tight_layout()
plt.show()

Grocery (in-store) dominates at 15.93%, nearly double the next 
category. The top 5 categories account for **51.4%** of all platform 
spending, concentrated in everyday essentials and retail. `grocery_net` 
is the smallest at 2.79%, likely because most grocery shopping happens 
in-store rather than online.

## 7. Transaction Amount Distribution

Before aggregating to monthly totals, we examine the shape of 
individual transaction amounts. Heavy right-skew or extreme outliers 
at the transaction level can create unstable variance in monthly 
aggregates — which would violate SARIMA's assumptions and may 
require a log transformation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full range
axes[0].hist(df['amt'], bins=100, color='steelblue', edgecolor='none')
axes[0].axvline(df['amt'].mean(),   color='red',    linestyle='--', linewidth=1.5, label=f"Mean:   ${df['amt'].mean():.2f}")
axes[0].axvline(df['amt'].median(), color='orange', linestyle='--', linewidth=1.5, label=f"Median: ${df['amt'].median():.2f}")
axes[0].set_title('Transaction Amounts — Full Range', fontsize=13)
axes[0].set_xlabel('Amount ($)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Zoomed to 95th percentile
p95 = df['amt'].quantile(0.95)
axes[1].hist(df['amt'][df['amt'] <= p95], bins=100, color='steelblue', edgecolor='none')
axes[1].set_title(f'Transaction Amounts — Zoomed to 95th Pct (≤ ${p95:.0f})', fontsize=13)
axes[1].set_xlabel('Amount ($)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

skewness = df['amt'].skew()
print(f"Skewness:                   {skewness:.2f}")
print(f"% under $100:               {(df['amt'] < 100).mean()*100:.1f}%")
print(f"% under $500:               {(df['amt'] < 500).mean()*100:.1f}%")
print(f"% over $1,000:              {(df['amt'] > 1000).mean()*100:.1f}%")
print(f"95th percentile:            ${p95:,.2f}")
print(f"Max transaction:            ${df['amt'].max():,.2f}")

The distribution is **extremely right-skewed (skewness: 45.37)**, 
driven entirely by a small number of large transactions. However, 
**82.2% of transactions are under $100** and **99.1% are under $500**, 
meaning the vast majority of spending is small everyday purchases. 
Only 0.2% of transactions exceed $1,000, and the max is $28,948.

The 95th percentile sits at just $189.59, confirming the extreme 
values are genuine outliers rather than a fat tail. Because we are 
forecasting **monthly aggregates** (summing tens of thousands of 
transactions per month), the Central Limit Theorem will stabilize 
the distribution considerably. We will assess whether a log 
transform is needed after inspecting the monthly series in Section 8.

## 8. Monthly Trend — Aggregate Series

We aggregate all transactions into monthly totals, producing a 
**24-point time series** from Jan 2019 to Dec 2020. This is the 
primary series we will model with SARIMA and Prophet. We look for 
visible trend and seasonality — particularly holiday spending spikes 
in December — which would justify a seasonal model.

In [ ]:
# Create year-month period column and aggregate
df['month'] = df['trans_date_trans_time'].dt.to_period('M')
monthly = df.groupby('month')['amt'].sum().reset_index()
monthly.columns = ['month', 'total_spending']
monthly['month_dt'] = monthly['month'].dt.to_timestamp()  # convert Period → datetime for plotting

# Rolling average
monthly['rolling_3'] = monthly['total_spending'].rolling(window=3, center=True).mean()

# Plot
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly['month_dt'], monthly['total_spending'], marker='o', linewidth=2, label='Monthly Total')
ax.plot(monthly['month_dt'], monthly['rolling_3'],      linewidth=2, linestyle='--', color='orange', label='3-Month Rolling Avg')

# Annotate Decembers
for _, row in monthly[monthly['month_dt'].dt.month == 12].iterrows():
    ax.annotate('Dec', xy=(row['month_dt'], row['total_spending']),
                xytext=(0, 12), textcoords='offset points', ha='center', fontsize=9, color='red')

ax.set_title('Monthly Total Spending — All Users (2019–2020)', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Total Spending ($)')
ax.legend()
plt.tight_layout()
plt.show()

# Print full table
print(monthly[['month', 'total_spending']].to_string(index=False))

### Interpretation — Monthly Trend: Aggregate Series

The data reveals a **clear and consistent seasonal pattern** that 
repeats identically in both 2019 and 2020:

- **January–February**: spending drops sharply (~$3.3–3.5M) — 
  post-holiday pullback
- **March–May**: moderate recovery (~$4.5–4.9M)
- **June–August**: sustained summer peak (~$5.8–5.9M)
- **September–November**: dip back to mid-range (~$4.6–4.9M)
- **December**: massive holiday spike — **$9.58M in 2019, $9.46M in 2020** — 
  roughly **2x the average non-December month**

There is **no meaningful upward or downward trend** across the two 
years — the series is level outside of seasonality. The December 
spike is so large it will be the dominant signal in any seasonal 
model. This pattern strongly justifies **SARIMA with period=12**, 
and the stability across both years suggests the seasonal component 
is reliable and not a one-off anomaly.

## 9. Seasonal Decomposition — Aggregate Series

We formally separate the monthly spending series into its **trend, 
seasonal, and residual components** using additive decomposition. 
This is the primary justification for using SARIMA — if the 
decomposition shows clean, repeating seasonality and a stable 
residual, the series is well-suited for a seasonal model.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

monthly_indexed = monthly.set_index('month_dt')['total_spending']

decomp = seasonal_decompose(monthly_indexed, model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 10))

decomp.observed.plot(ax=axes[0], title='Observed')
decomp.trend.plot(ax=axes[1],    title='Trend')
decomp.seasonal.plot(ax=axes[2], title='Seasonal')
decomp.resid.plot(ax=axes[3],    title='Residual')  # fixed

for ax in axes:
    ax.set_xlabel('')

plt.suptitle('Seasonal Decomposition — Monthly Aggregate Spending', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print("Seasonal component by month:")
print(decomp.seasonal.round(0).to_string())

### Interpretation — Seasonal Decomposition

The decomposition confirms everything we saw in Section 8, now 
formally separated into components:

**Trend:** No meaningful upward or downward drift across 2019–2020. 
The series is level, meaning spending behavior is stable year-over-year. 
Note that the first and last few months show `NaN` in the trend — 
this is expected with additive decomposition, not an error.

**Seasonal component:** The pattern repeats identically in both years, 
confirming it is a true structural signal:

- **December**: +$4,385,096 above baseline — by far the dominant effect
- **February**: -$2,012,124 — deepest post-holiday trough  
- **January**: -$1,717,417 — second weakest month
- **June–August**: modest positive (+$660K–$720K) — consistent summer bump
- **September–November**: slight negative drag (~-$450K to -$593K)

**Residual:** The residual is nearly flat — a direct consequence of 
having only **2 complete seasonal cycles** (24 months, period=12). 
With exactly two repetitions, the additive decomposition fits the 
seasonal component almost perfectly, leaving minimal residual 
variation. This is a data length limitation, not a modeling problem. 
In practice it means we should interpret SARIMA confidence intervals 
conservatively, as the model has limited data to estimate uncertainty.

## 10. Category-Level Monthly Trends

We break down monthly spending by all 14 categories to understand 
whether they move together or independently. This determines whether 
a single aggregate model is sufficient or whether category-level 
models are warranted in later phases.

In [ ]:
# Monthly spending per category
df['month'] = df['trans_date_trans_time'].dt.to_period('M')
cat_monthly = (
    df.groupby(['month', 'category'])['amt']
    .sum()
    .reset_index()
)
cat_monthly['month_dt'] = cat_monthly['month'].dt.to_timestamp()

# Pivot to wide format: rows = months, columns = categories
cat_pivot = cat_monthly.pivot(index='month_dt', columns='category', values='amt')

# --- Plot 1: All 14 categories on one chart ---
fig, ax = plt.subplots(figsize=(14, 6))
for col in cat_pivot.columns:
    ax.plot(cat_pivot.index, cat_pivot[col], linewidth=1.5, label=col)
ax.set_title('Monthly Spending by Category — All 14 Categories', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Total Spending ($)')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

# --- Plot 2: Top 5 categories separately ---
top5 = (
    df.groupby('category')['amt']
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .index.tolist()
)

fig, axes = plt.subplots(5, 1, figsize=(14, 18), sharex=True)
for ax, cat in zip(axes, top5):
    ax.plot(cat_pivot.index, cat_pivot[cat], marker='o', linewidth=2, color='steelblue')
    ax.set_title(cat, fontsize=11)
    ax.set_ylabel('Spending ($)')
plt.suptitle('Monthly Spending — Top 5 Categories', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# --- Plot 3: Correlation heatmap ---
fig, ax = plt.subplots(figsize=(12, 10))
corr = cat_pivot.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, annot_kws={'size': 8})
ax.set_title('Correlation Between Category Monthly Spending Series', fontsize=13)
plt.tight_layout()
plt.show()

# Print correlation matrix
print(corr.round(2).to_string())

Every category correlation is between **0.92 and 1.00** — essentially 
perfect. All 14 categories rise and fall in lockstep, driven entirely 
by the same seasonal pattern we identified in Section 9. This is a 
direct consequence of the synthetic data generation — spending was 
simulated with a shared seasonal signal applied uniformly across 
all categories.

**Modeling implication:** There is no meaningful information gained 
by modeling categories separately. A VAR model (which captures 
interdependencies between series) is unnecessary — the categories 
are not just correlated, they are virtually identical signals at 
different scales. We will model the **aggregate series only** with 
SARIMA and Prophet. Category-level forecasting is deprioritized 
for this dataset.

`travel` is the only slight outlier at 0.92–0.95, likely because 
it has higher variance from larger individual transactions — but 
even this is not meaningfully different.

## 11. User-Level Completeness Analysis

For individual-level forecasting with SARIMA, we need users with 
a **complete, uninterrupted monthly history**. A user who skips 
months cannot be modeled with a standard time series approach. 
We assess how many of our 908 users actually meet this requirement.

In [ ]:
# Count distinct months per user
user_months = (
    df.groupby('cc_num')['month']
    .nunique()
    .reset_index()
    .rename(columns={'month': 'num_months'})
)

total_months = df['month'].nunique()
complete_users = user_months[user_months['num_months'] == total_months]

print(f'Total months in dataset:        {total_months}')
print(f'Total users:                    {user_months.shape[0]}')
print(f'Users with all {total_months} months:         {len(complete_users)}')
print(f'Users with < {total_months} months:           {user_months.shape[0] - len(complete_users)}')
print()
print('Months per user distribution:')
print(user_months['num_months'].describe().round(1))

# Histogram
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(user_months['num_months'], bins=total_months, color='steelblue', edgecolor='white')
ax.axvline(total_months, color='red', linestyle='--', linewidth=1.5,
           label=f'Complete ({total_months} months)')
ax.set_title('Distribution of Active Months per User', fontsize=13)
ax.set_xlabel('Number of Months with Transactions')
ax.set_ylabel('Number of Users')
ax.legend()
plt.tight_layout()
plt.show()

Every single one of the 908 users has transactions in all 24 months 
— zero gaps, zero variance. This is another fingerprint of synthetic 
data, but it is **ideal for our modeling purposes**. Every user is 
a valid candidate for individual-level SARIMA forecasting with no 
imputation or gap-filling required.

In real consumer data we would expect significant dropout — users 
closing accounts, switching cards, or simply having inactive months. 
We would note this limitation in the README.

## 12. Representative User Selection + User EDA

We select one user whose monthly spending pattern is closest to the 
**median** across all users — not the highest or lowest spender, but 
the most typical. This user becomes our individual-level SARIMA proof 
of concept carried through all modeling notebooks. We also build a 
full user-level monthly table across all 908 users, which becomes 
the input to our XGBoost model in a later phase.

In [ ]:
# --- Build full user-level monthly table (all 908 users) ---
monthly_all_users = (
    df.groupby(['cc_num', 'month'])['amt']
    .sum()
    .reset_index()
    .rename(columns={'amt': 'total_spending'})
)
monthly_all_users['month_dt'] = monthly_all_users['month'].dt.to_timestamp()

print('Full user-level monthly table shape:', monthly_all_users.shape)
print('Expected: 908 users × 24 months =', 908 * 24)
print()

# --- Select representative user ---
# Compute each user's total spending over 24 months
user_totals = (
    monthly_all_users.groupby('cc_num')['total_spending']
    .sum()
    .reset_index()
    .rename(columns={'total_spending': 'total_24mo'})
)

# Find user closest to median total spending
median_total = user_totals['total_24mo'].median()
user_totals['dist_from_median'] = (user_totals['total_24mo'] - median_total).abs()
rep_user = user_totals.loc[user_totals['dist_from_median'].idxmin(), 'cc_num']

print(f'Median 24-month spending across all users: ${median_total:,.2f}')
print(f'Representative user cc_num:                {rep_user}')
print(f'Their 24-month total:                      ${user_totals.loc[user_totals["cc_num"] == rep_user, "total_24mo"].values[0]:,.2f}')

The full user-level monthly table is confirmed at **21,792 rows** 
(908 users × 24 months) — this is the XGBoost input table we will 
engineer lag features from in a later phase.

User **639030014711** is selected as our representative individual 
with a 24-month total of **$129,616.64** against a median of 
**$129,571.32** — a difference of only $45, making them essentially 
the median user. This user carries through all individual-level 
modeling as our SARIMA proof of concept.

## 13. Representative User EDA

We examine the spending pattern of our selected representative user 
(cc_num: 639030014711) in detail — monthly trend, category breakdown, 
and seasonal decomposition. We then compare their monthly spending 
to the platform aggregate to confirm they are truly typical and not 
an outlier in disguise.

In [ ]:
# --- Filter to representative user ---
rep_monthly = monthly_all_users[monthly_all_users['cc_num'] == rep_user].copy()

# --- Plot 1: Monthly spending trend ---
rep_monthly['rolling_3'] = rep_monthly['total_spending'].rolling(window=3, center=True).mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(rep_monthly['month_dt'], rep_monthly['total_spending'],
        marker='o', linewidth=2, label='Monthly Spending')
ax.plot(rep_monthly['month_dt'], rep_monthly['rolling_3'],
        linewidth=2, linestyle='--', color='orange', label='3-Month Rolling Avg')
ax.set_title(f'Monthly Spending — User {rep_user}', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Spending ($)')
ax.legend()
plt.tight_layout()
plt.show()

# --- Plot 2: Category breakdown for this user ---
user_cats = (
    df[df['cc_num'] == rep_user]
    .groupby('category')['amt']
    .sum()
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(user_cats.index, user_cats.values, color='steelblue')
ax.set_title(f'Spending by Category — User {rep_user}', fontsize=13)
ax.set_xlabel('Total Spending ($)')
plt.tight_layout()
plt.show()

# --- Plot 3: Seasonal decomposition for this user ---
from statsmodels.tsa.seasonal import seasonal_decompose

rep_indexed = rep_monthly.set_index('month_dt')['total_spending']
rep_decomp = seasonal_decompose(rep_indexed, model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 10))
rep_decomp.observed.plot(ax=axes[0], title='Observed')
rep_decomp.trend.plot(ax=axes[1],    title='Trend')
rep_decomp.seasonal.plot(ax=axes[2], title='Seasonal')
rep_decomp.resid.plot(ax=axes[3],    title='Residual')
for ax in axes:
    ax.set_xlabel('')
plt.suptitle(f'Seasonal Decomposition — User {rep_user}', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# --- Compare user vs aggregate ---
print('User vs Aggregate Monthly Comparison:')
print(f'{"Month":<12} {"User ($)":>12} {"Aggregate ($)":>15} {"User % of Agg":>15}')
print('-' * 56)
for _, row in rep_monthly.iterrows():
    agg_val = monthly.loc[monthly['month_dt'] == row['month_dt'], 'total_spending'].values[0]
    pct = row['total_spending'] / agg_val * 100
    print(f'{str(row["month"]):<12} {row["total_spending"]:>12,.0f} {agg_val:>15,.0f} {pct:>14.2f}%')

### Interpretation — Representative User EDA

The user's December spikes ($11,721 in 2019, $10,160 in 2020) mirror 
the aggregate pattern exactly, confirming they exhibit the same 
seasonal behavior as the platform. Their monthly spending is 
consistently **0.08–0.15% of the aggregate**, which is exactly what 
you'd expect from 1 of 908 users (theoretical share: 0.11%).

The **residual is flat** for the same reason as the aggregate — only 
2 seasonal cycles means the additive decomposition fits perfectly, 
leaving nothing in the residual. This is a data limitation, not a 
modeling problem, and applies to every user in this dataset.

One notable pattern: this user shows **more month-to-month variance** 
than the aggregate (individual spending is noisier than the sum of 
908 users), which means individual-level SARIMA confidence intervals 
will be wider than aggregate-level ones. This is expected and 
important to communicate in the modeling notebooks.

## 14. Save Processed Files

We save three datasets that feed directly into the modeling notebooks. 
Each file serves a distinct purpose — aggregate forecasting, 
individual proof-of-concept forecasting, and the full user-level 
table for XGBoost feature engineering.

In [ ]:
import os

os.makedirs('../data/processed', exist_ok=True)

# 1. Full cleaned transaction table — XGBoost feature engineering input
df.drop(columns=['month']).to_csv('../data/processed/transactions_clean.csv', index=False)
print(f'transactions_clean.csv saved:      {df.shape[0]:,} rows × {df.shape[1]-1} cols')

# 2. Aggregate monthly series — SARIMA/Prophet input
monthly[['month_dt', 'total_spending']].to_csv('../data/processed/monthly_spending.csv', index=False)
print(f'monthly_spending.csv saved:        {monthly.shape[0]} rows × 2 cols')

# 3. Representative user monthly series — individual SARIMA input
rep_monthly[['month_dt', 'total_spending']].to_csv(
    f'../data/processed/monthly_user_{rep_user}.csv', index=False
)
print(f'monthly_user_{rep_user}.csv saved: {rep_monthly.shape[0]} rows × 2 cols')

# 4. All users monthly table — XGBoost forecasting input
monthly_all_users[['cc_num', 'month_dt', 'total_spending']].to_csv(
    '../data/processed/monthly_all_users.csv', index=False
)
print(f'monthly_all_users.csv saved:       {monthly_all_users.shape[0]:,} rows × 3 cols')

print()
print('All files saved to ../data/processed/')
print(f'Representative user cc_num: {rep_user}')
print('Hardcode this value in all modeling notebooks.')

## 15. Key Findings & EDA Summary

A complete summary of what the data revealed and how each finding 
drives modeling decisions in subsequent notebooks.

### Dataset & Quality
- Combined dataset: **1,842,743 legitimate transactions** across 
  **908 users, 14 categories, 693 merchants** from Jan 2019 – Dec 2020
- Fraud rate was 0.52% — removed entirely as we model legitimate 
  spending behavior only
- Zero missing values, zero users with incomplete monthly histories 
  — every user has all 24 months, making all 908 users valid 
  candidates for individual-level forecasting

### Spending Patterns
- **grocery_pos dominates** at 15.93% of total platform spending — 
  nearly double the next category
- Top 5 categories (grocery_pos, shopping_pos, gas_transport, home, 
  shopping_net) account for **51.4% of all spending**
- Individual transactions are heavily right-skewed 
  (skewness: 45.37) but **82.2% are under $100** — monthly 
  aggregates are stable due to high transaction volume

### Seasonality & Trend
- The aggregate monthly series shows a **strong, consistent seasonal 
  pattern** repeating identically in 2019 and 2020
- **December is the dominant effect**: +$4.39M above baseline, 
  roughly 2x an average month
- **February is the weakest month**: -$2.01M below baseline — 
  post-holiday pullback
- **No trend**: spending is level year-over-year, confirming the 
  series is driven by seasonality not growth
- Seasonal decomposition confirms clean decomposable structure, 
  directly justifying **SARIMA with period=12**

### Category Independence
- All 14 category correlations fall between **0.92 and 1.00** — 
  categories move in perfect lockstep driven by the same seasonal 
  signal
- **Modeling implication**: no information gain from modeling 
  categories separately. Aggregate-level forecasting is sufficient. 
  VAR modeling is unnecessary.

### User-Level Analysis
- All 908 users have complete 24-month histories — no imputation needed
- Representative user **639030014711** selected with 24-month total 
  of $129,616 vs median of $129,571 — effectively the median user
- Individual user series shows the same seasonal pattern as the 
  aggregate but with **higher month-to-month variance**, meaning 
  individual-level confidence intervals will be wider than 
  aggregate-level ones

### Modeling Decisions Locked In
| Decision | Justification |
|---|---|
| SARIMA with period=12 | Clean seasonal decomposition, 12-month cycle confirmed |
| Additive decomposition | No multiplicative growth in trend |
| Aggregate + individual user modeling | Both series valid, different variance profiles |
| No category-level modeling | Correlations 0.92–1.00, no independent signal |
| Log transform — defer decision | Individual transactions skewed but monthly aggregates stable — assess in SARIMA notebook |
| XGBoost on all 908 users | Complete histories enable lag feature engineering for all users |

### Limitations
- Data is **synthetic** — seasonal pattern repeats with unrealistic 
  precision. Real consumer data would exhibit irregular residuals, 
  trend shifts, and external shocks (e.g. COVID-19 in 2020)
- Only **2 complete seasonal cycles** — SARIMA parameter estimation 
  and confidence intervals should be interpreted conservatively
- Flat residuals in decomposition are a consequence of data length, 
  not a modeling problem
- In production, this pipeline would ingest real transaction data 
  via **Plaid API**, where these limitations would not apply

## 16. Files Saved to `data/processed/`

| File | Rows | Columns | Used In |
|---|---|---|---|
| `transactions_clean.csv` | 1,842,743 | 12 | XGBoost feature engineering |
| `monthly_spending.csv` | 24 | 2 | SARIMA, Prophet — aggregate series |
| `monthly_user_639030014711.csv` | 24 | 2 | SARIMA, Prophet — individual series |
| `monthly_all_users.csv` | 21,792 | 3 | XGBoost — all users monthly totals |

**Hardcoded values for all modeling notebooks:**
- Representative user: `cc_num = 639030014711`
- Forecast horizon: 6 months
- Seasonal period: 12
- Time-based train/test split: first 18 months train, last 6 months test